# Does feature regularisation hold on other datasets?

**Where this comes from.** On *credit*, a convex logistic regression on out-of-bag-honest tree bits matched LightGBM, and *feature* regularisation (deleting deep layers, flipping bits) cut the train-minus-validation gap more than tenfold with no cost in test AUC. The advisor's response, point by point, and how this notebook answers it:

| the advisor said | what this notebook does |
|---|---|
| *try it on a few more datasets* | six new datasets plus credit as the anchor, across **binary, multi-class and regression** |
| *keep all experimental factors intact* | identical forest (100 trees, depth 6, min-leaf 5), identical OOB protocol, identical head strength (C = 0.01; Ridge alpha = 1/(2C) for regression), identical arms, identical noise copies, identical row cap, identical seed |
| *track all metrics across train, val and test* | every run reports AUC, accuracy and log-loss (or RMSE, R² and MAE) on **all three splits**; train is measured on clean bits |
| *report that train accuracy fell from 1 (no overfit)* | the verdict table prints train and test side by side for the baseline and for the keep-depths-0-2 arm, plus the gap |
| *feature regularisation instead of model regularisation, see if it holds* | the same eight arms on every dataset, one verdict row per dataset with an explicit yes/no |

### Datasets
| type | dataset | OpenML task | rows | features | classes |
|---|---|---|---|---|---|
| binary, anchor | credit | 361055 | 16,714 → 16,000 | 10 | 2 |
| binary | house_16H | 361063 | 13,488 | 16 | 2 |
| binary | MagicTelescope | 361065 | 13,376 | 10 | 2 |
| multi-class | pendigits | 32 | 10,992 | 16 | 10 |
| multi-class | covertype | 7593 | 581,012 → 16,000 | 54 | 7 |
| regression | cpu_act | 361072 | 8,192 | 21 | — |
| regression | wine_quality | 361076 | 6,497 | 11 | — |

The multi-class head is multinomial logistic regression and AUC is macro one-vs-rest. The regression head is Ridge with alpha = 1/(2C) = 50, the same L2 coefficient relative to the summed loss as the classification setting; R² is the primary metric and RMSE/MAE are on the original scale.

### Arms, identical on every dataset
| arm | what it does |
|---|---|
| baseline | all bits, no perturbation (also runs the raw-features floor) |
| uniform p=0.2 | the noise level that was best on credit |
| ramp P=0.2 | depth-increasing flips |
| uniform p=0.16 | the matched-budget control for that ramp |
| drop depth 5 | delete the deepest layer |
| drop depths 4,5 | delete the two deepest layers |
| drop 4,5 + ramp 0.2 | the combined design |
| **keep depths 0-2** | the configuration recommended from credit |

**Observation holds** on a dataset if the keep-depths-0-2 arm is within 0.01 of the baseline on the test metric **and** has a smaller train-minus-validation gap.

⏱ **roughly 1.5–2 hours, CPU only.** The suite is resumable: rerunning the cell skips finished (dataset, arm) pairs, so a dropped session costs at most one run. Runtime → Run all.

In [ ]:
# 1 · mount Drive + get the code
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, shutil
DRIVE = '/content/drive/MyDrive/TKCE/featreg_suite'
os.makedirs(DRIVE, exist_ok=True)
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull
print('results ->', DRIVE)

In [ ]:
# 2 · install deps
!pip install -q openml catboost optuna lightgbm xgboost

In [ ]:
# 2b · OPTIONAL — only if OpenML 504s: upload openml_cache_clean5.tar.gz (else press Cancel)
#      (covers credit, house_16H, MagicTelescope; the others download from OpenML directly)
import tarfile
dst = '/root/.cache/openml/org/openml/www'
if glob.glob(dst + '/tasks/361055'):
    print('cache present — skip')
else:
    try:
        from google.colab import files; files.upload()
    except Exception as e: print('skipped:', e)
    hits = glob.glob('/content/**/openml_cache_clean5.tar.gz', recursive=True)
    if hits:
        os.makedirs(dst, exist_ok=True)
        with tarfile.open(hits[0]) as t: t.extractall(dst)
        print('cache extracted')
    else: print('no bundle — will use OpenML directly')

In [ ]:
# 3 · resume from Drive if an earlier session already finished some runs
if os.path.isdir(f'{DRIVE}/runs'):
    shutil.copytree(f'{DRIVE}/runs', 'results/featreg_suite', dirs_exist_ok=True)
    done = glob.glob('results/featreg_suite/*/*/featreg_*.json')
    print(f'restored {len(done)} finished runs from Drive; they will be skipped')
else:
    print('no earlier runs on Drive — starting fresh')

In [ ]:
# 4 · THE SUITE — 7 datasets x 8 arms, every factor held fixed, resumable
!python -u run_feature_reg_suite.py --root results/featreg_suite 2>&1 | tee results/featreg_suite/suite.log
shutil.copytree('results/featreg_suite', f'{DRIVE}/runs', dirs_exist_ok=True)
print('synced to Drive')

In [ ]:
# 5 · the verdict table again, on its own
import pandas as pd
pd.set_option('display.width', 250)
v = pd.read_csv('results/featreg_suite/suite_verdict.csv')
print(v.to_string(index=False))
print(f"\nobservation holds on {int((v.holds == 'yes').sum())} of {len(v)} datasets")

In [ ]:
# 6 · 'train accuracy fell from 1': baseline vs keep-depths-0-2, ALL metrics, ALL splits
t = pd.read_csv('results/featreg_suite/suite_long.csv')
for ds in dict.fromkeys(t.dataset):
    g = t[(t.dataset == ds) & (t.view == 'tree') & (t.arm.isin(['00_baseline', '40_keep_L012']))]
    prim = g.primary.iloc[0]
    sec = [c for c in g.columns if c.startswith(('train_', 'val_', 'test_'))]
    print(f'\n=== {ds} ({g.task_type.iloc[0]}, primary = {prim}) ===')
    for _, r in g.iterrows():
        print(f"  {r.arm:14s} bits={int(r.bits):5d}  "
              f"train {r.train:.4f}  val {r.val:.4f}  test {r.test:.4f}  gap {r.gap:+.4f}")
        extra = {c: r[c] for c in sec}
        print('                 ' + '  '.join(f'{k}={v:.4f}' for k, v in extra.items()))

In [ ]:
# 7 · the cross-dataset picture
from IPython.display import Image, display
display(Image('results/featreg_suite/suite_overview.png'))
shutil.copy('results/featreg_suite/suite_overview.png', f'{DRIVE}/suite_overview.png')
shutil.copy('results/featreg_suite/suite_verdict.csv', f'{DRIVE}/suite_verdict.csv')
shutil.copy('results/featreg_suite/suite_long.csv', f'{DRIVE}/suite_long.csv')
print('tables and figure on Drive:', DRIVE)

In [ ]:
# 8 · every arm, every dataset — the full long table
cols = ['dataset','task_type','arm','view','bits','mean_p','train','val','test','gap','tree_ceiling']
print(t[cols].round(4).to_string(index=False))